---
**Copyright 2026 Nicola Vessio - Tutti i diritti riservati.**

Questo software (notebook, codice e relativa documentazione) è opera di **Nicola Vessio**
ed è protetto dalle norme vigenti in materia di diritto d'autore (L. 633/1941 e successive
modifiche, Convenzione di Berna, Direttiva 2009/24/CE sulla tutela giuridica dei programmi
per elaboratore).

Sono riservati all'autore tutti i diritti di utilizzazione economica dell'opera, inclusi
a titolo esemplificativo la riproduzione, la distribuzione, la modifica, l'adattamento,
la traduzione e la comunicazione al pubblico. **Ogni uso non espressamente autorizzato
per iscritto dall'autore è vietato.**

_Contatto: da definire._

---

# Report delle operazioni MT5 (da file) - Ausilio alla dichiarazione dei redditi

Il presente notebook legge un report di cronistoria esportato da MetaTrader 5
(in formato Excel .xlsx) ed elabora le operazioni in esso contenute, allo scopo di
produrre un report chiaro e ordinato a supporto della dichiarazione dei redditi.

A differenza della versione "live", questo modulo NON si collega al terminale MT5:
lavora su un file già esportato, quindi non richiede che l'applicazione sia aperta.
È adatto a chi non può o non vuole tenere il terminale aperto, o deve elaborare
un estratto fornito da terzi.

## Come esportare ed usare il file da MetaTrader 5

Nel terminale MT5, nella scheda "Cronistoria" (Storico conto), fare clic destro e
scegliere "Report" -> "Foglio di calcolo XML di Office 2007". Questa voce genera un
file con estensione .xlsx, che è il file da fornire a questo notebook.

Nota: viene accettato SOLO il formato .xlsx. Il formato "Report HTML" non è supportato.

Per il corretto funzionamento, *il file generato da MetaTrader5 (ReportHistory...xlsx) deve essere NELLA STESSA CARTELLA di questo notebook.*

## Conti supportati

Lo strumento elabora il report di un conto MetaTrader 5, sia esso un conto operativo
diretto sia un conto in copytrading, presso qualsiasi broker.

## Funzionalità

- Lettura del report .xlsx esportato da MT5
- Estrazione delle operazioni complete, già abbinate da MT5 (apertura + chiusura)
- Separazione tra operazioni di trading e movimenti di cassa (depositi/prelievi)
- Lettura automatica dei dati del conto dall'intestazione del file
- Calcolo del profitto netto dei costi del broker (commissioni + swap)
- Produzione di un report Excel a quattro fogli

## Valuta del conto

Gli importi sono espressi nella valuta del conto (tipicamente USD). L'eventuale
conversione in euro ai fini della dichiarazione dei redditi italiana non è gestita
qui ed è a cura dello studio commercialistico.

## Prerequisiti

- Python 3.12 (ambiente Miniconda consigliato)
- Pacchetti: pandas, openpyxl
- Il file .xlsx esportato da MT5 (vedi istruzioni sopra)

## Nota importante

Questo notebook è uno strumento di organizzazione ed esposizione dei dati, non una
consulenza fiscale. I dati vanno verificati e la dichiarazione è gestita dal
commercialista o CAF. Il regime fiscale applicabile, le eventuali compensazioni, la
corretta imputazione di plusvalenze e minusvalenze, la conversione in euro, il quadro RW
e l'imposta di bollo sono di competenza dello studio commercialistico.

---

## LIBRERIE

Tutte le librerie usate nel notebook, raccolte in un punto solo. Questa cella va eseguita per prima: le celle successive danno per scontato che questi import siano già caricati.

In [ ]:
import glob                                                              # individuare i file nella cartella
import os                                                                # percorsi e nomi file

import pandas as pd                                                      # gestione dei dati in tabelle

from datetime import datetime                                            # interpretare le date delle operazioni

from openpyxl import load_workbook                                       # leggere il file MT5
from openpyxl import Workbook                                            # creare il report Excel
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side   # stili del report
from openpyxl.utils import get_column_letter                             # larghezza colonne

---

## IMPORTAZIONE DATI

Il primo passo è portare dentro il notebook il report esportato da MetaTrader 5 e comprenderne la struttura, così da sapere dove si trovano le informazioni utili.

Il report MT5 è un unico foglio di calcolo organizzato in sezioni successive (operazioni, ordini, movimenti). In questa fase individuiamo il file nella cartella, lo apriamo e localizziamo il punto in cui inizia ciascuna sezione: le celle di estrazione successive sapranno così esattamente dove leggere.

### Selezione del report

Il notebook cerca automaticamente i report esportati da MT5 nella cartella in cui si trova (i file che iniziano con "ReportHistory"). Non occorre indicare nomi o percorsi: basta collocare il file .xlsx lì dentro.

**Non rinominare** il file esportato da MT5: deve mantenere il nome originale che inizia con "ReportHistory", altrimenti il notebook non riesce a individuarlo.

Se nella cartella è presente un solo report, viene usato direttamente. Se ce n'è più di uno (ad esempio conti su broker diversi), il notebook chiede quale analizzare: viene mostrato l'elenco numerato e si digita il numero corrispondente.

> **Nota per VS Code:** quando viene richiesto il numero del report, la casella in cui digitare compare **in alto**, nella barra dei comandi in cima alla finestra, non sotto la cella.

In [ ]:
# Elenchiamo i report trovati nella cartella (file "ReportHistory*.xlsx")
file_trovati = sorted(glob.glob("ReportHistory*.xlsx"))

if not file_trovati:
    raise SystemExit(
        "Nessun file trovato. Collocare il report .xlsx esportato da MT5 "
        "(nome originale tipo 'ReportHistory-XXXXXXXX.xlsx', da NON rinominare) "
        "nella cartella del notebook."
    )

# Se c'e' un solo report, lo usiamo direttamente.
# Se ce n'e' piu' d'uno, chiediamo quale analizzare (l'esecuzione si ferma in attesa).
if len(file_trovati) == 1:
    percorso_file = file_trovati[0]
else:
    print("Trovati piu' report nella cartella:")
    for i, f in enumerate(file_trovati, start=1):
        print(f"  {i}. {f}")

    # Chiediamo il numero finche' non ne arriva uno valido
    while True:
        scelta = input(f"Quale report analizzare? (1-{len(file_trovati)}): ")
        if scelta.isdigit() and 1 <= int(scelta) <= len(file_trovati):
            percorso_file = file_trovati[int(scelta) - 1]
            break
        print(f"  Scelta non valida: digitare un numero da 1 a {len(file_trovati)}.")

print(f"Report selezionato: {percorso_file}")

### Lettura del file e sezioni

Il report MT5 è un unico foglio diviso in sezioni, ciascuna introdotta da una riga-titolo. Qui il foglio viene caricato e si individua a quale riga inizia ciascuna sezione, così le celle successive sanno dove leggere:

- **Posizioni**: le operazioni già abbinate da MT5 (apertura + chiusura)
- **Ordini**: gli ordini immessi (non usati nell'analisi)
- **Affari**: da cui si ricavano i movimenti di cassa (depositi, prelievi, performance fee)

In [ ]:
# Apriamo il file scelto nella cella precedente.
# data_only=True legge i valori (non le eventuali formule).
wb = load_workbook(percorso_file, data_only=True)
ws = wb.active   # il report MT5 ha un unico foglio

# Scorriamo la prima colonna alla ricerca delle righe-titolo di sezione,
# salvando il numero di riga in cui inizia ciascuna.
riga_posizioni = None
riga_ordini    = None
riga_affari    = None

for riga in range(1, ws.max_row + 1):
    valore = ws.cell(row=riga, column=1).value
    if valore == "Posizioni":
        riga_posizioni = riga
    elif valore == "Ordini":
        riga_ordini = riga
    elif valore == "Affari":
        riga_affari = riga

# Se non troviamo le sezioni attese, il file non e' un report MT5 valido.
if riga_posizioni is None or riga_affari is None:
    raise SystemExit(
        "Il file non sembra un report di cronistoria MT5 valido "
        "(sezioni 'Posizioni'/'Affari' non trovate). Verificare di aver esportato "
        "il file corretto da MetaTrader 5."
    )

print("Sezioni individuate:")
print(f"  'Posizioni' inizia alla riga {riga_posizioni}")
print(f"  'Ordini'    inizia alla riga {riga_ordini}")
print(f"  'Affari'    inizia alla riga {riga_affari}")

**Nota sull'avviso "UserWarning" di openpyxl**

Durante la lettura del file, Python potrebbe mostrare un avviso giallo simile a:

`UserWarning: Workbook contains no default style, apply openpyxl's default`

Non è un errore e non compromette in alcun modo i dati o il report. Segnala semplicemente che il file esportato da MetaTrader 5 non contiene uno stile grafico predefinito (i report MT5 sono privi di formattazione): la libreria di lettura ne applica uno proprio e prosegue normalmente. L'elaborazione e i calcoli non sono influenzati, l'avviso può essere ignorato.

---

## ESTRAZIONE E LETTURA DEI DATI

Una volta individuata la struttura del file, ne ricaviamo le informazioni che servono all'analisi, tralasciando tutto il resto. Il report MT5 contiene infatti più dati di quelli utili ai fini fiscali: qui selezioniamo solo ciò che conta e lo organizziamo in tabelle ordinate.

In particolare:

- **le operazioni**: i trade completi già abbinati da MT5 (apertura e chiusura di ogni posizione), con profitto, commissioni e swap;
- **i movimenti di cassa**: depositi, prelievi ed eventuali altri movimenti di denaro, tenuti separati dai trade perché non generano plusvalenze o minusvalenze;
- **i dati del conto**: le informazioni anagrafiche (intestatario, broker, numero di conto, valuta) lette dall'intestazione del file, necessarie per identificare il report.

Vengono invece tralasciati i dati non rilevanti per l'analisi, come gli ordini non eseguiti.

### Estrazione delle operazioni

La sezione "Posizioni" contiene le operazioni già abbinate da MT5: ogni riga è un trade completo (apertura + chiusura) con il relativo profitto. Costruiamo un DataFrame con le stesse colonne del modulo live, così il report Excel finale ha struttura identica.

Le colonne del report MT5 sono, nell'ordine:

- **Ora apertura** - data e ora di apertura della posizione
- **Posizione** - identificativo univoco del trade
- **Simbolo** - strumento negoziato (es. XAUUSD)
- **Tipo** - direzione dell'operazione (buy o sell)
- **Volume** - dimensione in lotti
- **Prezzo apertura** - prezzo di ingresso
- **S/L e T/P** - livelli di stop loss e take profit
- **Ora chiusura** - data e ora di chiusura
- **Prezzo chiusura** - prezzo di uscita
- **Commissioni** - costo trattenuto dal broker
- **Swap** - interesse per il mantenimento overnight
- **Profitto** - risultato lordo dei costi

In [ ]:
# I dati delle Posizioni iniziano 2 righe dopo il titolo (titolo + intestazione)
# e finiscono 1 riga prima del titolo "Ordini".
prima_riga_dati = riga_posizioni + 2
ultima_riga_dati = riga_ordini - 1

trade_completi = []
for riga in range(prima_riga_dati, ultima_riga_dati + 1):
    valori = [ws.cell(row=riga, column=c).value for c in range(1, 14)]

    # Salta eventuali righe vuote
    if valori[0] is None:
        continue

    ora_apertura   = valori[0]    # datetime apertura
    position_id    = valori[1]    # id posizione
    simbolo        = valori[2]    # es. XAUUSD
    tipo           = valori[3]    # 'buy' o 'sell'
    volume         = valori[4]
    ora_chiusura   = valori[8]    # datetime chiusura
    commissione    = valori[10] if valori[10] is not None else 0.0
    swap           = valori[11] if valori[11] is not None else 0.0
    profitto       = valori[12] if valori[12] is not None else 0.0

    profit_netto = profitto + swap + commissione

    trade_completi.append({
        "position_id":   position_id,
        "symbol":        simbolo,
        "volume":        volume,
        "tipo":          str(tipo).upper(),                 # BUY / SELL
        "data_apertura": ora_apertura,
        "data_chiusura": ora_chiusura,
        "profit_lordo":  round(profitto, 2),
        "swap":          round(swap, 2),
        "commission":    round(commissione, 2),
        "profit_netto_costi_broker_lordo_imposte_USD":
            round(profit_netto, 2),
    })

df_trade = pd.DataFrame(trade_completi).sort_values("data_chiusura").reset_index(drop=True)

print(f"Operazioni estratte: {len(df_trade)}")
print(f"Profitto netto totale (USD): {df_trade['profit_netto_costi_broker_lordo_imposte_USD'].sum():.2f}")
print()
print(df_trade.head(12))

### Estrazione dei movimenti di cassa

Depositi, prelievi e performance fee non compaiono tra le "Posizioni" (che sono solo trade): si trovano nella sezione "Affari", nelle righe di tipo "balance". Sono movimenti di denaro, non operazioni di trading, e vanno tenuti separati perché non generano plusvalenze o minusvalenze.

Ogni movimento viene classificato in una di quattro categorie, in base al segno dell'importo e alla descrizione:

- **Deposito**: importo positivo (versamento di capitale sul conto)
- **Performance fee**: importo negativo con descrizione "split" (il compenso trattenuto dal servizio di copytrading)
- **Prelievo**: importo negativo con descrizione "withdraw" (trasferimento verso il conto bancario)
- **Altro**: importo negativo senza le due parole chiave (movimento da verificare, che non viene comunque perso)

Nella sezione "Affari" i dati utili si leggono da tre colonne: il **Tipo** (colonna 4, dove compare "balance"), il **Profitto** (colonna 12, che contiene l'importo del movimento) e il **Commento** (colonna 14, la descrizione).

In [ ]:
# I dati degli Affari iniziano 2 righe dopo il titolo (titolo + intestazione)
prima_riga_affari = riga_affari + 2

movimenti_cassa = []
for riga in range(prima_riga_affari, ws.max_row + 1):
    tipo = ws.cell(row=riga, column=4).value   # colonna "Tipo"

    # Ci fermiamo quando finiscono i dati: se anche l'ora (col 1) e' vuota, usciamo
    if ws.cell(row=riga, column=1).value is None:
        break

    # Ci interessano SOLO le righe di tipo "balance" (movimenti di cassa)
    if tipo is not None and str(tipo).lower() == "balance":
        ora      = ws.cell(row=riga, column=1).value    # data/ora del movimento
        affare   = ws.cell(row=riga, column=2).value    # id del movimento
        importo  = ws.cell(row=riga, column=12).value   # colonna "Profitto" = importo
        commento = ws.cell(row=riga, column=14).value   # descrizione del movimento

        importo = importo if importo is not None else 0.0
        descrizione = str(commento).lower() if commento is not None else ""

        # Classifichiamo il movimento in base al segno e alla descrizione.
        # L'ordine conta: prima il deposito (positivo), poi le parole chiave.
        # "split" identifica la performance fee del servizio di copytrading.
        if importo > 0:
            categoria = "Deposito"
        elif "split" in descrizione:
            categoria = "Performance fee"
        elif "withdraw" in descrizione:
            categoria = "Prelievo"
        else:
            categoria = "Altro"

        movimenti_cassa.append({
            "id":        affare,
            "data":      ora,
            "importo":   round(importo, 2),
            "categoria": categoria,
            "commento":  commento if commento is not None else "",
        })

# Riepilogo di controllo a schermo, per categoria
print(f"Movimenti di cassa trovati: {len(movimenti_cassa)}")
for m in movimenti_cassa:
    print(f"  {m['data']} | {m['categoria']:16} | {m['importo']:>10.2f} USD | {m['commento']}")

### Lettura dei dati del conto

I dati anagrafici del conto (intestatario, broker, numero, valuta) si trovano nell'intestazione del report, nelle prime righe del foglio. Li leggiamo per identificare il conto nel report Excel finale.

Il numero di conto è scritto su una riga unica insieme ad altri dati, nel formato `NUMERO (VALUTA, SERVER, TIPO, MODALITA)` - ad esempio `12345678 (USD, BrokerServer-Live, real, Hedge)`. La riga viene quindi scomposta nei singoli campi.

In [ ]:
# Leggiamo i valori grezzi dall'intestazione (colonna 4)
nome_intestatario = ws.cell(row=2, column=4).value or "n/d"
riga_conto        = ws.cell(row=3, column=4).value or ""
societa           = ws.cell(row=4, column=4).value or "n/d"

# Spacchettiamo la riga del conto. Formato: "NUMERO (VALUTA, SERVER, TIPO, MODALITA)"
numero_conto = "n/d"
valuta       = "n/d"
server       = "n/d"
tipo_conto   = "n/d"
modalita     = "n/d"

if riga_conto:
    if "(" in riga_conto:
        # Prima della parentesi: il numero conto
        numero_conto = riga_conto.split("(")[0].strip()
        # Tra parentesi: valuta, server, tipo, modalita' (separati da virgola)
        dentro_parentesi = riga_conto.split("(")[1].rstrip(")")
        parti = [p.strip() for p in dentro_parentesi.split(",")]
        if len(parti) >= 1:
            valuta = parti[0]
        if len(parti) >= 2:
            server = parti[1]
        if len(parti) >= 3:
            tipo_conto = parti[2]
        if len(parti) >= 4:
            modalita = parti[3]
    else:
        numero_conto = riga_conto.strip()

# Raccogliamo tutto in un dizionario (come "info" del modulo live)
info_conto = {
    "intestatario": nome_intestatario,
    "societa":      societa,
    "numero_conto": numero_conto,
    "server":       server,
    "valuta":       valuta,
    "tipo_conto":   tipo_conto,
    "modalita":     modalita,
}

print("Dati del conto letti dal file:")
for chiave, valore in info_conto.items():
    print(f"  {chiave}: {valore}")

In [ ]:
# ============================================================
# Generazione del report Excel
# ============================================================
# Crea un file .xlsx con QUATTRO fogli:
#   1) Operazioni        -> una riga per ogni trade (le posizioni chiuse)
#   2) Riepilogo         -> dati del conto + totali fiscali (con FORMULE Excel vere)
#   3) Movimenti di cassa-> depositi/prelievi, tenuti separati dai trade
#   4) Guida e glossario -> spiegazioni (BUY/SELL, forex, termini) a corredo
# I totali del Riepilogo sono formule: se si modifica una riga, si ricalcolano da soli.

# --- Movimenti di cassa: sono gia' una lista di dizionari pronti ---
df_cassa = pd.DataFrame(movimenti_cassa)

# Ultima riga dei movimenti di cassa (per i totali nel Riepilogo).
# I dati partono da riga 4; se non ci sono movimenti resta 3 (solo intestazione).
ultima_riga_cassa = len(df_cassa) + 3 if not df_cassa.empty else 3

# ============================================================
# Periodo coperto dal report (dal primo all'ultimo movimento)
# ============================================================
# Ricaviamo l'intervallo di date guardando TUTTE le date presenti: sia le
# operazioni (apertura e chiusura) sia i movimenti di cassa. Cosi' il periodo
# copre anche eventuali movimenti (prelievi, performance fee) successivi
# all'ultimo trade. Le date sono nel formato AAAA.MM.GG, che si ordina
# correttamente anche come testo.

date_report = []

if not df_trade.empty:
    date_report += [str(d) for d in df_trade["data_apertura"]]
    date_report += [str(d) for d in df_trade["data_chiusura"]]

if not df_cassa.empty:
    date_report += [str(d) for d in df_cassa["data"]]

if date_report:
    data_prima = min(date_report)
    data_ultima = max(date_report)
    periodo_dal = data_prima[:10]
    periodo_al  = data_ultima[:10]
else:
    periodo_dal = periodo_al = "n/d"

# --- Stili riutilizzabili ---
BLU        = "1F4E78"
GRIGIO     = "D9D9D9"
ZEBRA      = "EEF3F8"
VERDE_TXT  = "1E7B34"
ROSSO_TXT  = "C0392B"
font_tit   = Font(name="Arial", size=14, bold=True, color="1F4E78")
font_head  = Font(name="Arial", size=11, bold=True, color="FFFFFF")
font_norm  = Font(name="Arial", size=10)
font_bold  = Font(name="Arial", size=10, bold=True)
font_verde = Font(name="Arial", size=10, color=VERDE_TXT)
font_rosso = Font(name="Arial", size=10, color=ROSSO_TXT)
fill_head  = PatternFill("solid", fgColor=BLU)
fill_tot   = PatternFill("solid", fgColor=GRIGIO)
fill_zebra = PatternFill("solid", fgColor=ZEBRA)
bordo      = Border(*[Side(style="thin", color="BFBFBF")]*4)
centro     = Alignment(horizontal="center", vertical="center")
sinistra   = Alignment(horizontal="left", vertical="top", wrap_text=True)
sx_centro  = Alignment(horizontal="left", vertical="center")

def scrivi_intestazioni(ws, intestazioni, riga=1):
    for col, testo in enumerate(intestazioni, start=1):
        c = ws.cell(row=riga, column=col, value=testo)
        c.font = font_head
        c.fill = fill_head
        c.alignment = centro
        c.border = bordo

def banda_titolo(ws, riga, testo, n_colonne):
    for col in range(1, n_colonne + 1):
        c = ws.cell(row=riga, column=col)
        c.fill = fill_head
        c.border = bordo
        if col == 1:
            c.value = testo
            c.font = font_head
            c.alignment = sx_centro

# ============================================================
# FOGLIO 1 - OPERAZIONI
# ============================================================
wb = Workbook()
ws1 = wb.active
ws1.title = "Operazioni"

intest_op = [
    "N.", "ID Posizione", "Strumento", "Volume (lotti)", "Tipo",
    "Data apertura", "Data chiusura",
    "Profit lordo (USD)", "Swap (USD)", "Commissione (USD)",
    "Profit netto costi broker - lordo imposte (USD)",
]
scrivi_intestazioni(ws1, intest_op)

for i, (_, r) in enumerate(df_trade.iterrows(), start=1):
    riga = i + 1
    netto = r["profit_netto_costi_broker_lordo_imposte_USD"]
    riga_colorata = (i % 2 == 0)
    valori = [
        i, r["position_id"], r["symbol"], r["volume"], r["tipo"],
        str(r["data_apertura"]), str(r["data_chiusura"]),
        r["profit_lordo"], r["swap"], r["commission"], netto,
    ]
    for col, v in enumerate(valori, start=1):
        c = ws1.cell(row=riga, column=col, value=v)
        c.font = font_norm
        c.border = bordo
        if riga_colorata:
            c.fill = fill_zebra
        if col in (8, 9, 10, 11):
            c.number_format = '#,##0.00'
        if col == 11:
            c.font = font_verde if netto >= 0 else font_rosso

ultima_riga_op = len(df_trade) + 1
ws1.freeze_panes = "A2"

larghezze_op = [5, 14, 11, 13, 8, 20, 20, 16, 12, 14, 30]
for col, w in enumerate(larghezze_op, start=1):
    ws1.column_dimensions[get_column_letter(col)].width = w

# ============================================================
# FOGLIO 2 - RIEPILOGO (dati del conto + totali con formule)
# ============================================================
ws2 = wb.create_sheet("Riepilogo")

col_netto = "K"
rng_netto = f"Operazioni!{col_netto}2:{col_netto}{ultima_riga_op}"

# --- Banda: DATI DEL CONTO ---
banda_titolo(ws2, 1, "DATI DEL CONTO", 2)

dati_conto = [
    ("Intestatario",       info_conto["intestatario"]),
    ("Broker",             info_conto["societa"]),
    ("Numero conto",       str(info_conto["numero_conto"])),
    ("Server",             info_conto["server"]),
    ("Valuta",             info_conto["valuta"]),
    ("Tipo conto",         info_conto["tipo_conto"]),
    ("Modalita'",          info_conto["modalita"]),
    ("Periodo delle operazioni", f"dal {periodo_dal} al {periodo_al}"),
]

r = 2
for i, (etichetta, valore) in enumerate(dati_conto):
    ce = ws2.cell(row=r, column=1, value=etichetta)
    cv = ws2.cell(row=r, column=2, value=valore)
    ce.font = font_bold
    cv.font = font_norm
    ce.border = bordo
    cv.border = bordo
    if i % 2 == 1:
        ce.fill = fill_zebra
        cv.fill = fill_zebra
    r += 1

# --- Banda: RIEPILOGO FISCALE ---
r += 1
banda_titolo(ws2, r, "RIEPILOGO FISCALE", 2)
r += 1

righe_riep = [
    ("Numero di operazioni (trade)",        f'=COUNT({rng_netto})'),
    ("Operazioni in guadagno (plus)",       f'=COUNTIF({rng_netto},">0")'),
    ("Operazioni in perdita (minus)",       f'=COUNTIF({rng_netto},"<0")'),
    ("Somma plusvalenze nette (USD)",       f'=SUMIF({rng_netto},">0")'),
    ("Somma minusvalenze nette (USD)",      f'=SUMIF({rng_netto},"<0")'),
    ("RISULTATO NETTO COMPLESSIVO (USD)",   f'=SUM({rng_netto})'),
    ("Totale commissioni (USD)",            f'=SUM(Operazioni!J2:J{ultima_riga_op})'),
    ("Totale swap (USD)",                   f'=SUM(Operazioni!I2:I{ultima_riga_op})'),
]

for etichetta, formula in righe_riep:
    ws2.cell(row=r, column=1, value=etichetta).font = font_bold
    c = ws2.cell(row=r, column=2, value=formula)
    c.font = font_norm
    c.number_format = '#,##0.00'
    if "COMPLESSIVO" in etichetta:
        ws2.cell(row=r, column=1).fill = fill_tot
        c.fill = fill_tot
        c.font = font_bold
    r += 1

# --- Banda: MOVIMENTI DI CASSA (totali per categoria) ---
r += 1
banda_titolo(ws2, r, "MOVIMENTI DI CASSA", 2)
r += 1

col_cat = f"'Movimenti di cassa'!D4:D{ultima_riga_cassa}"
col_imp = f"'Movimenti di cassa'!C4:C{ultima_riga_cassa}"

righe_cassa = [
    ("Totale depositi (USD)",        f'=SUMIF({col_cat},"Deposito",{col_imp})'),
    ("Totale prelievi (USD)",        f'=SUMIF({col_cat},"Prelievo",{col_imp})'),
    ("Totale performance fee (USD)", f'=SUMIF({col_cat},"Performance fee",{col_imp})'),
    ("RISULTATO NETTO DOPO PERFORMANCE FEE (USD)", "=B17+B24"),
]

for i, (etichetta, formula) in enumerate(righe_cassa):
    ce = ws2.cell(row=r, column=1, value=etichetta)
    c  = ws2.cell(row=r, column=2, value=formula)
    ce.font = font_bold
    c.font = font_norm
    c.number_format = '#,##0.00'
    ce.border = bordo
    c.border = bordo
    if "DOPO PERFORMANCE FEE" in etichetta:
        # Riga chiave evidenziata come il risultato netto complessivo
        ce.fill = fill_tot
        c.fill = fill_tot
        c.font = font_bold
    elif i % 2 == 1:
        ce.fill = fill_zebra
        c.fill = fill_zebra
    r += 1

r += 1
ws2.cell(row=r, column=1,
         value="Tutti gli importi sono in USD. Conversione in EUR, quadro RW e "
               "imposta di bollo a cura dello studio commercialistico.").font = font_norm

ws2.column_dimensions["A"].width = 42
ws2.column_dimensions["B"].width = 26

# ============================================================
# FOGLIO 3 - MOVIMENTI DI CASSA
# ============================================================
ws3 = wb.create_sheet("Movimenti di cassa")
ws3["A1"] = ("Depositi e prelievi - NON sono operazioni di trading, "
             "non generano plusvalenze/minusvalenze")
ws3["A1"].font = font_bold

intest_cassa = ["ID", "Data", "Importo (USD)", "Categoria", "Descrizione"]
scrivi_intestazioni(ws3, intest_cassa, riga=3)

if not df_cassa.empty:
    for i, (_, r_) in enumerate(df_cassa.iterrows(), start=1):
        riga = i + 3
        riga_colorata = (i % 2 == 0)
        valori = [
            r_["id"], str(r_["data"]), r_["importo"], r_["categoria"], r_["commento"],
        ]
        for col, v in enumerate(valori, start=1):
            c = ws3.cell(row=riga, column=col, value=v)
            c.font = font_norm
            c.border = bordo
            if riga_colorata:
                c.fill = fill_zebra
            if col == 3:
                c.number_format = '#,##0.00'

ws3.freeze_panes = "A4"

for col, w in enumerate([12, 20, 14, 12, 30], start=1):
    ws3.column_dimensions[get_column_letter(col)].width = w

# ============================================================
# FOGLIO 4 - GUIDA E GLOSSARIO
# ============================================================
ws4 = wb.create_sheet("Guida e glossario")

banda_titolo(ws4, 1, "COME LEGGERE QUESTO REPORT", 2)

paragrafi = [
    'Questo report riguarda operazioni di trading su strumenti finanziari derivati (CFD) '
    'effettuate tramite due broker esteri (ad esempio TMGM - TradeMax Global e FPG - Fortune Prime Global).',
    '"BUY" e "SELL" NON sono acquisti o vendite di beni o valute reali. Sono contratti (CFD) '
    'che replicano la variazione di prezzo di uno strumento: non si possiede mai il bene o la '
    'valuta sottostante. BUY = puntata sul rialzo (posizione lunga); SELL = puntata sul ribasso '
    '(posizione corta). Il risultato e\' solo un importo in denaro (USD).',
    'FOREX (coppie di valute): strumenti come EURUSD, GBPUSD, USDJPY. Si opera sulla variazione '
    'del tasso di cambio, non sullo scambio fisico delle valute. XAUUSD indica oro/dollaro, '
    'anch\'esso trattato come strumento e mai consegnato fisicamente.',
    'VALUTA: il conto e\' denominato in dollari USA (USD). Tutti gli importi sono in USD. '
    'La conversione in euro e\' a cura dello studio commercialistico.',
    'BROKER ESTERI: quando il broker ha sede estera, gli adempimenti collegati '
    '(quadro RW, imposta di bollo) sono curati dallo studio commercialistico.',
]

r = 3
for p in paragrafi:
    c = ws4.cell(row=r, column=1, value=p)
    c.font = font_norm
    c.alignment = sinistra
    ws4.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
    ws4.row_dimensions[r].height = 45
    r += 2

r += 1
banda_titolo(ws4, r, "GLOSSARIO DEI TERMINI", 2)
r += 1
scrivi_intestazioni(ws4, ["Termine", "Significato"], riga=r)
r += 1

glossario = [
    ("CFD", "Contratto che replica il prezzo di uno strumento senza possederlo. Risultato in denaro."),
    ("Forex", "Mercato dei tassi di cambio, strumenti espressi come coppie di valute."),
    ("Coppia di valute", "Es. EURUSD: prima valuta 'base', seconda 'quotata'. Si opera sul cambio."),
    ("BUY (posizione lunga)", "Puntata sul rialzo del prezzo. Non e' acquisto di beni/valute reali."),
    ("SELL (posizione corta)", "Puntata sul ribasso del prezzo. Non e' vendita di beni/valute reali."),
    ("Operazione (trade)", "Operazione completa: apertura + chiusura, gia' abbinate da MT5."),
    ("Position ID", "Codice che identifica una singola operazione."),
    ("Volume (lotti)", "Dimensione dell'operazione. Non indica quantita' di merce/valuta posseduta."),
    ("Symbol (strumento)", "Nome dello strumento (es. EURUSD, GBPUSD, XAUUSD)."),
    ("Profit lordo", "Risultato dell'operazione prima dei costi del broker. In USD."),
    ("Commission", "Costo trattenuto dal broker. Valore negativo. In USD."),
    ("Swap", "Interesse per il mantenimento della posizione oltre la giornata. Di norma negativo. In USD."),
    ("Profit netto costi broker", "Risultato dopo commissioni e swap. Al lordo delle imposte. In USD."),
    ("Movimento di cassa", "Deposito o prelievo. Non e' trading, non genera plus/minusvalenze."),
    ("Performance fee", "Compenso trattenuto dal gestore del segnale di copytrading, calcolato sui risultati del periodo. "
    "Viene addebitato come movimento separato sul conto, di norma all'inizio del mese successivo a quello di riferimento: "
    "la fee visibile a inizio mese si riferisce quindi al mese precedente. Importo in USD. "
    "Il relativo trattamento fiscale e' di competenza dello studio commercialistico."),
    ("Plusvalenza", "Guadagno complessivo realizzato."),
    ("Minusvalenza", "Perdita complessiva realizzata."),
    ("Quadro RW", "Sezione per il monitoraggio delle attivita' finanziarie estere. A cura dello studio."),
    ("Imposta di bollo", "Imposta annua sui conti oltre soglie di giacenza. A cura dello studio."),
]

for i, (termine, significato) in enumerate(glossario):
    ct = ws4.cell(row=r, column=1, value=termine)
    cs = ws4.cell(row=r, column=2, value=significato)
    ct.font = font_bold
    cs.font = font_norm
    ct.alignment = sx_centro
    cs.alignment = sinistra
    ct.border = bordo
    cs.border = bordo
    if i % 2 == 1:
        ct.fill = fill_zebra
        cs.fill = fill_zebra
    r += 1

ws4.column_dimensions["A"].width = 30
ws4.column_dimensions["B"].width = 85

# ============================================================
# SALVATAGGIO
# ============================================================
# --- Nome broker automatico, letto dal file ---
# Usa il nome societa' completo (es. "TradeMax Global Limited"), togliendo le
# parole generiche finali (Limited, Ltd, ecc.) che non identificano il broker,
# e sostituendo gli spazi con trattini per renderlo adatto a un nome file.
# Se il nome non e' disponibile, usa un ripiego generico.
company = str(info_conto["societa"]).strip()

if company and company.lower() != "n/d":
    for generica in ["Limited", "Ltd", "LLC", "Inc", "Incorporated", "Group", "Pty", "S.A.", "SA", "SpA"]:
        company = company.replace(generica, "")
    broker = "-".join(company.split())
    if not broker:
        broker = "BROKER"
else:
    broker = "BROKER"

# Periodo per il nome file: usiamo le date gia' calcolate per il report,
# con i punti sostituiti da trattini (piu' adatti a un nome di file).
if periodo_dal != "n/d":
    dal_file = periodo_dal.replace(".", "-")
    al_file  = periodo_al.replace(".", "-")
    nome_file = f"report_trading_{broker}_{dal_file}_{al_file}_da-file.xlsx"
else:
    nome_file = f"report_trading_{broker}_periodo-n-d_da-file.xlsx"

wb.save(nome_file)
print(f"Report salvato: {nome_file}")
print(f"  - Broker rilevato: {info_conto['societa']}")
print(f"  - Periodo: dal {periodo_dal} al {periodo_al}")
print(f"  - Operazioni: {len(df_trade)} trade")
print(f"  - Movimenti di cassa: {len(df_cassa)} movimenti")
print("  - IMPORTANTE: aprire e salvare in Excel prima di inviare, per calcolare i totali.")

## APPENDICE - Come leggere il report

Il report riguarda operazioni di trading su strumenti finanziari derivati (CFD)
effettuate tramite broker online (ad esempio TMGM - TradeMax Global e FPG - Fortune
Prime Global). Si chiariscono i punti che generano più spesso confusione.

### "BUY" e "SELL" NON sono acquisti o vendite di beni o valute reali

Le diciture BUY (compra) o SELL (vendi) su strumenti come EURUSD, GBPUSD, USDJPY o
XAUUSD non indicano l'acquisto o la vendita fisica di valuta estera, oro o altre merci.

Si tratta di contratti finanziari (CFD - Contract For Difference): contratti che
replicano la variazione di prezzo di uno strumento. Il bene o la valuta sottostante
non viene mai posseduto. Non vengono ricevuti dollari, sterline, yen o lingotti d'oro:
esiste solo un profitto o una perdita in denaro (in USD), in base all'andamento del prezzo.

- BUY = posizione al rialzo ("lunga"): si punta sull'aumento del prezzo
- SELL = posizione al ribasso ("corta"): si punta sulla diminuzione del prezzo

In entrambi i casi il risultato è solo un importo in denaro (in USD), non un bene fisico.

### Cosa sono le coppie di valute (Forex)

Il Forex (Foreign Exchange) è il mercato dei tassi di cambio. Gli strumenti si
presentano come coppie di valute, es. EURUSD: la prima valuta (EUR) è la "base",
la seconda (USD) è la "quotata".

Operare su una coppia significa puntare sulla variazione del tasso di cambio, non
scambiare fisicamente le due valute. Alcuni esempi di coppie:

- EURUSD - Euro / Dollaro USA
- GBPUSD - Sterlina britannica / Dollaro USA
- USDJPY - Dollaro USA / Yen giapponese
- XAUUSD - Oro / Dollaro USA

### Tutti gli importi sono in dollari USA (USD)

Il conto di trading è denominato in USD. Ogni cifra del report (profitti, perdite,
commissioni) è espressa in dollari. La conversione in euro è a cura dello studio
commercialistico, ai cambi di riferimento.

### Broker esteri

Quando i broker hanno sede estera (come nel caso di TMGM e FPG), possono sussistere
adempimenti specifici (es. quadro RW per il monitoraggio delle attività finanziarie
detenute all'estero, ed eventuale imposta di bollo), di competenza dello studio
commercialistico.

### In sintesi

Non vi è stato alcun acquisto o vendita di valute o merci. Sono stati aperti e chiusi
contratti finanziari il cui unico esito è stato un guadagno o una perdita in denaro
(USD), riepilogati nel presente documento.